# Authorized E12 seed42 saved research
Context256, accumulation4, exact5000 authoritative update stop. Source e1f0c9861fcdd38d07cd24e746900d53a314b740. Private input syedarijaali/e12-seed42-verified-smoke-recovery-20261013 contains byte-identical independently retrieved qualification output. First cell is development recovery; second launches the sole research run and canonical analysis. No other research configuration/seed/extension.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT=Path('/tmp/e12_source'); OUT=Path('/kaggle/working')
SOURCE='e1f0c9861fcdd38d07cd24e746900d53a314b740'
env={**os.environ,'UV_PYTHON_INSTALL_DIR':'/tmp/e12_python','UV_CACHE_DIR':'/tmp/e12_cache','UV_NATIVE_TLS':'true','CUBLAS_WORKSPACE_CONFIG':':4096:8','OMP_NUM_THREADS':'1','MPLCONFIGDIR':'/tmp/e12_mpl','MPLBACKEND':'Agg','NANO_SOURCE_COMMIT':SOURCE,'NANO_SESSION_LIMIT_SECONDS':'43200'}
def command(args,cwd=None):
    print('EXEC',args,flush=True)
    with (OUT/'bootstrap.log').open('a') as log:
        result=subprocess.run(args,cwd=cwd,env=env,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((OUT/'bootstrap.log').read_text()[-4000:],flush=True)
        raise RuntimeError('E12 gate failed; preserve output')
assert not ROOT.exists()
command(['git','clone','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(ROOT)])
command(['git','checkout','--detach',SOURCE],ROOT)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=ROOT,text=True).strip()
command([sys.executable,'-m','pip','install','--target','/tmp/e12_tools','uv==0.12.7'])
UV=Path('/tmp/e12_tools/bin/uv')
command([str(UV),'python','install','3.11.14'])
command([str(UV),'venv','--seed','--python','3.11.14','/tmp/e12_venv'])
PY=Path('/tmp/e12_venv/bin/python')
freeze=(ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock=OUT/'qualified-lock.txt';lock.write_text(chr(10).join(x for x in freeze if not x.startswith('-e '))+chr(10))
command([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'])
command([str(UV),'pip','install','--python',str(PY),'-r',str(lock)])
command([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'])
command([str(PY),'-m','pip','check'])
archives=list(Path('/kaggle/input').rglob('E12_smoke_bundle.bin'))
assert len(archives)==1
archive=archives[0];manifest=archive.parent/'E12_smoke_manifest.json'
import hashlib
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='47f045f5eb3ef0388102830b02fd62ae5b07372ff532cbcdfae1d5fbf69e6568'
command([str(PY),str(ROOT/'scripts/pilot/e12_kaggle.py'),'recover-smoke','--root',str(ROOT),'--output',str(OUT/'persistent_recovery'),'--archive',str(archive),'--manifest',str(manifest)],ROOT)
print((OUT/'persistent_recovery/persistent_recovery.json').read_text())


In [ ]:
gate=json.loads((OUT/'persistent_recovery/persistent_recovery.json').read_text())
assert gate['status']=='saved_output_archive_and_fresh_compatible_CUDA_full_state_recovery_passed'
assert gate['additional_research_optimizer_updates']==0
assert not (OUT/'research').exists() and not (OUT/'durable_staging').exists()
print('E12 RESEARCH GO: seed42, context256, accumulation4, exact stop5000',flush=True)
research_started=time.time()
subprocess.run([str(PY),str(ROOT/'scripts/pilot/e12_kaggle.py'),'research','--root',str(ROOT),'--output',str(OUT/'research'),'--store',str(OUT/'durable_staging'),'--recovery-receipt',str(OUT/'persistent_recovery/persistent_recovery.json')],cwd=ROOT,env=env,check=True)
subprocess.run([str(PY),str(ROOT/'scripts/pilot/e12_kaggle.py'),'finish','--root',str(ROOT),'--output',str(OUT/'research'),'--store',str(OUT/'durable_staging')],cwd=ROOT,env=env,check=True)
(OUT/'research/pipeline_observed.json').write_text(json.dumps({'research_and_finish_seconds':time.time()-research_started,'scope':'Research,validation,IO,generation,verification and local packaging; excludes bootstrap,smoke,platformpublication and independentdownload'}))
print('E12 STOPPED AT5000; SAVED PUBLICATION AND INDEPENDENT FINAL RECOVERY STILL REQUIRED',flush=True)
